<a href="https://colab.research.google.com/github/Rodin2/E-commerce-system/blob/main/NileByte_Systems_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NileByte Systems — RAG Pipeline (Keyword Overlap + Stopwords)

## نبذة عن الشركة
**NileByte Systems** هي شركة برمجيات وتقنية معلومات تعتمد على سياسات داخلية واضحة للموارد البشرية والعمل (مثل العمل عن بُعد، مواعيد العمل، صرف الرواتب، البدلات، السفر، والإجازات)[cite: 1].

## الغرض من النوتبوك
تهدف هذه النوتبوك إلى بناء وتجربة نظام **RAG** للإجابة بدقة على أسئلة موظفي **NileByte Systems** استنادًا إلى قاعدة المعرفة الداخلية للشركة، ومقارنة أداؤه بإجابة الموديل بدون سياق (`Without RAG`)[cite: 1].

### ما تم تنفيذه في النوتبوك
1. **قاعدة المعرفة (`Knowledge Base`):** تضم 7 سياسات داخلية خاصة بالشركة[cite: 1].
2. **الاسترجاع بالكلمات المفتاحية (`Keyword Retriever`):** تحويل النصوص إلى مجموعة كلمات صغيرة (`lowercase`) بعد استبعاد الكلمات الشائعة (`STOPWORDS`) واسم الشركة المتكرر، ثم ترتيب المستندات حسب عدد الكلمات المشتركة مع السؤال[cite: 1].
3. **معالجة الأسئلة المركبة (`top_k=3`):** رفع عدد المستندات المسترجعة من `2` إلى `3` ساعد على استرجاع المعلومات الخاصة بشقي السؤال المركّب معًا (أيام العمل عن بُعد وموعد تحويل الرواتب)[cite: 1].
4. **توليد الإجابة مع `Fallback`:** تجربة أكثر من موديل متاح في الحساب (`gemini-3.5-flash-lite` ثم `gemini-3.5-flash` ثم `gemini-3.6-flash`) لتقليل أخطاء الضغط أو الانشغال[cite: 1].

In [1]:
# Step 1: تجهيز قاعدة المعرفة (Knowledge Base) لشركة NileByte Systems
knowledge_base = [
    "NileByte Systems allows employees to work remotely (Work From Home) two days a week: strictly on Sundays and Wednesdays.",
    "New employees at NileByte Systems receive an annual leave balance of 21 days, which becomes active only after completing a 3-month probation period.",
    "Official working hours at the NileByte Systems headquarters are from 9:00 AM to 5:00 PM, including a 1-hour lunch break.",
    "Salaries at NileByte Systems are transferred on the 28th of every month. If the 28th falls on a weekend, salaries are paid on the preceding Thursday.",
    "To claim the monthly internet and laptop allowance ($50), NileByte Systems employees must submit a request on the internal HR portal before the 15th of the month.",
    "Business travel policy at NileByte Systems covers economy flight tickets and hotel accommodation up to a maximum of $150 per night.",
    "Sick leave at NileByte Systems requires a medical certificate from an approved clinic if the absence exceeds two consecutive days."
]

print(f"Knowledge Base ready with {len(knowledge_base)} documents.")

Knowledge Base ready with 7 documents.


In [2]:
# Step 2: تهيئة عمل Gemini API
import time
import numpy as np
from google import genai
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [3]:
# Step 3: بناء Semantic Retriever باستخدام gemini-embedding-001
EMBEDDING_MODEL = "gemini-embedding-001"

def get_embedding(text: str) -> np.ndarray:
    res = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
    )
    vec = np.array(res.embeddings[0].values, dtype=np.float32)
    return vec / np.linalg.norm(vec)

# حساب المتجهات لقاعدة المعرفة مرة واحدة
kb_embeddings = np.vstack([get_embedding(doc) for doc in knowledge_base])
print(f"Embeddings generated with shape: {kb_embeddings.shape}")

def retrieve_semantic(query: str, kb: list[str], kb_vecs: np.ndarray, top_k: int = 3, min_score: float = 0.35):
    query_vec = get_embedding(query)
    scores = kb_vecs @ query_vec
    ranked_indices = np.argsort(scores)[::-1]

    results = []
    for idx in ranked_indices[:top_k]:
        score = float(scores[idx])
        if score >= min_score:
            results.append((idx + 1, kb[idx], score))
    return results

Embeddings generated with shape: (7, 3072)


In [4]:
# Step 4: بناء Augmented Prompt مع Citations + توليد الإجابة مع Fallback
def build_augmented_prompt(query: str, retrieved_items: list[tuple[int, str, float]]) -> str:
    if not retrieved_items:
        context_block = "No relevant policy documents found."
    else:
        context_block = "\n".join(
            [f"[Doc {doc_num}] (similarity: {score:.2f}) {doc_text}" for doc_num, doc_text, score in retrieved_items]
        )

    return f"""You are a precise HR assistant for NileByte Systems.
Answer the user's question strictly using ONLY the provided policy documents below.
- Cite the source document number in square brackets after every claim, e.g., [Doc 1].
- Do not infer, extrapolate, or add external details.
- If the answer cannot be found in the provided documents, respond ONLY with:
"I don't have information about that in NileByte Systems policies."

Provided Policy Documents:
{context_block}

User Question: {query}
Answer:"""

def generate_answer(prompt: str) -> str:
    models_to_try = [
        "gemini-3.5-flash-lite",
        "gemini-3.5-flash",
        "gemini-3.6-flash",
    ]
    for model_name in models_to_try:
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
            )
            if response.text:
                return response.text.strip()
        except Exception as e:
            print(f"⚠️ {model_name} unavailable/busy ({e}), trying next...")
            time.sleep(2)
    return "All configured models are currently unavailable. Please try again shortly."

In [5]:
# Step 5: تشغيل واختبار النظام على أسئلة متنوعة
test_questions = [
    "Which days am I allowed to work from home at NileByte Systems, and when are salaries transferred?",
    "How do I claim my internet allowance and what is the deadline?",
    "What is the maternity leave policy at NileByte Systems?"  # سؤال خارج السياسات لاختبار منع الهلوسة
]

for q in test_questions:
    print("=" * 70)
    print(f"❓ Question: {q}")
    retrieved = retrieve_semantic(q, knowledge_base, kb_embeddings, top_k=3)
    print("🔍 Retrieved:")
    for doc_num, doc_text, score in retrieved:
        print(f"   [Doc {doc_num}] ({score:.3f}): {doc_text}")
    prompt = build_augmented_prompt(q, retrieved)
    print("\n🤖 RAG Answer:")
    print(generate_answer(prompt))


❓ Question: Which days am I allowed to work from home at NileByte Systems, and when are salaries transferred?
🔍 Retrieved:
   [Doc 1] (0.866): NileByte Systems allows employees to work remotely (Work From Home) two days a week: strictly on Sundays and Wednesdays.
   [Doc 4] (0.801): Salaries at NileByte Systems are transferred on the 28th of every month. If the 28th falls on a weekend, salaries are paid on the preceding Thursday.
   [Doc 3] (0.784): Official working hours at the NileByte Systems headquarters are from 9:00 AM to 5:00 PM, including a 1-hour lunch break.

🤖 RAG Answer:
At NileByte Systems, employees are allowed to work remotely (Work From Home) strictly on Sundays and Wednesdays [Doc 1]. Salaries are transferred on the 28th of every month, or on the preceding Thursday if the 28th falls on a weekend [Doc 4].
❓ Question: How do I claim my internet allowance and what is the deadline?
🔍 Retrieved:
   [Doc 5] (0.720): To claim the monthly internet and laptop allowance ($50), N